# Preprocessing: Data Setup

Before RIME-X can emulate an indicator, that indicator has to go through a **preprocessing pipeline**. This notebook covers the first step of that pipeline: making RIME-X understand *where it gets the source data* it will later use for calibration. This source data is typically a multi-model-ensemble simulation of some climate indicator.

The preprocessing pipeline built into RIME-X is designed around **ISIMIP** indicators, but it can equally be applied to:
- indicators that are available locally (not through ISIMIP at all), or
- indicators that are derived/computed from other ISIMIP indicators.

This tutorial demonstrates all of this on three simple examples:

1. **Download an indicator directly from ISIMIP** — the indicator already exists in the ISIMIP repository and just needs downloading.
2. **Derive an indicator from ISIMIP data** — the indicator doesn't exist directly, but can be computed from one or more indicators that do.
3. **Use a local indicator** — the indicator's files already exist on disk (e.g. you preprocessed them yourself) and just need to be registered with RIME-X. Since RIME-X is conceived around the ISIMIP format, a local indicator may need some further preprocessing to approximately match it: one file per model/indicator/experiment combination, with `lon`, `lat`, and `time` dimensions, and the indicator itself stored as a data variable of the same name as the indicator.

You can browse the indicators that already exist in ISIMIP at [data.isimip.org](https://data.isimip.org).

This notebook walks through all three, using the example data provided in `rimeX/tutorial/data`.

**Note:** the following tutorials in this series build on the output of Option 3, so make sure to at least run that section before moving on. Running Options 1 and 2 will download around 2.5 GB from the ISIMIP repository into the data directory. 

In [8]:
import xarray as xr
from rimeX.datasets.download_isimip import CONFIG, Indicator

## Configuration

The first step is to set up the configuration. This can be done either in `rimeX/rimeX/config.toml`, or per-indicator directly in code — which is what we'll do here, so the tutorial is self-contained and works with the example data in `rimeX/tutorial/data`. If you're processing multiple indicators for a real project, we recommend setting these in `config.toml` instead.

The most important paths to configure are:
- `isimip.download_folder` — where raw indicator files get downloaded for options 1 and 2
- `indicators.folder` — where intermediate preprocessing outputs are saved
- `isimip.climate_impact_explorer` — where the final quantile maps (the last preprocessing step) are saved

If you're using ISIMIP3b data, the remaining defaults in `config.toml` should already work out of the box.

In [9]:
CONFIG['isimip.download_folder'] = 'data/downloads'
CONFIG['indicators.folder'] = 'data/indicators'
CONFIG['isimip.climate_impact_explorer'] = 'data'

## Defining indicators

Indicators can also be defined directly in `config.toml`, but we'll define them here for the tutorial.

- **Option 1** (download from ISIMIP): `temp-annual-max`
- **Option 2** (derived indicator): `temp-annual-max-degree-over-20`, computed from `temp-annual-max`

Let's define both.

In [10]:
# Option 1: this indicator already exists in the ISIMIP repository, so we only need to
# define it -- RIME-X will download it automatically.
# More examples of indicator definitions can be found in config.toml.
CONFIG['indicator.temp-annual-max'] = {
    'frequency': 'annual',
    'units': '°C',
    'isimip_meta': {
        'variable': 'temp-annual-max',
    }
}

# Option 2: this indicator doesn't exist in the ISIMIP repository, but depends on one that
# does. We declare the dependency via `depends_on`, and point `custom` at the function that
# computes it (defined here as `temp_annual_max_degree_over_20` in rimeX/indicators.py).
# More examples of such functions and indicator definitions can be found in rimeX/indicators.py
# and config.toml respectively.
CONFIG['indicator.temp-annual-max-degree-over-20'] = {
    'depends_on': ['temp-annual-max'],
    'frequency': 'annual',
    'custom': 'rimeX.indicators:temp_annual_max_degree_over_20',
}

## Downloading Option 1 & 2

With the config in place, we can build an `Indicator` object from it and download the underlying data with `run_download`. (If you defined your indicators in `config.toml` instead, this step is the same -- just make sure your environment is activated first.)

Alternatively, once your indicators are defined in `config.toml`, you don't need a notebook or Python code at all -- you can activate the environment RIME-X is installed in and download them directly from the command line with the `rime-download-isimip` tool, passing indicator names via `-i`/`--indicator`, e.g.:

```bash
rime-download-isimip -i temp-annual-max temp-annual-max-degree-over-20
```

Useful flags: `--overwrite` to redownload existing files, `--remove-daily` to discard intermediate daily files after aggregation, and `--download-folder` to override `isimip.download_folder` for this run.

In [4]:
temp_annual_max = Indicator.from_config("temp-annual-max")
temp_annual_max_degree_over_20 = Indicator.from_config("temp-annual-max-degree-over-20")

# !!!!!! This will download around 2.5 gb !!!!!
temp_annual_max.run_download(dry_run=False)
temp_annual_max_degree_over_20.run_download(dry_run=False)

[12:45:34 | rimeX | INFO] Saving 20 database entries to data/downloads/db/temp-annual-max.json
[12:45:34 | rimeX | INFO] [1/20] temp-annual-max: downloading {'climate_scenario': 'historical', 'climate_forcing': 'gfdl-esm4'} ...
[12:45:34 | rimeX | INFO] [1/20] temp-annual-max: done -> data/downloads/ISIMIP3b/DerivedInputData/SPARCCLE2025/historical/GFDL-ESM4/gfdl-esm4_r1i1p1f1_w5e5_historical_temp-annual-max_global_annual_1850_2014.nc
[12:45:34 | rimeX | INFO] [2/20] temp-annual-max: downloading {'climate_scenario': 'historical', 'climate_forcing': 'ipsl-cm6a-lr'} ...
[12:45:34 | rimeX | INFO] [2/20] temp-annual-max: done -> data/downloads/ISIMIP3b/DerivedInputData/SPARCCLE2025/historical/IPSL-CM6A-LR/ipsl-cm6a-lr_r1i1p1f1_w5e5_historical_temp-annual-max_global_annual_1850_2014.nc
[12:45:34 | rimeX | INFO] [3/20] temp-annual-max: downloading {'climate_scenario': 'historical', 'climate_forcing': 'mpi-esm1-2-hr'} ...
[12:45:34 | rimeX | INFO] [3/20] temp-annual-max: done -> data/download

[PosixPath('data/indicators/temp-annual-max-degree-over-20/historical/gfdl-esm4/gfdl-esm4_historical_temp-annual-max-degree-over-20_annual_1850_2014.nc'),
 PosixPath('data/indicators/temp-annual-max-degree-over-20/historical/ipsl-cm6a-lr/ipsl-cm6a-lr_historical_temp-annual-max-degree-over-20_annual_1850_2014.nc'),
 PosixPath('data/indicators/temp-annual-max-degree-over-20/historical/mpi-esm1-2-hr/mpi-esm1-2-hr_historical_temp-annual-max-degree-over-20_annual_1850_2014.nc'),
 PosixPath('data/indicators/temp-annual-max-degree-over-20/historical/mri-esm2-0/mri-esm2-0_historical_temp-annual-max-degree-over-20_annual_1850_2014.nc'),
 PosixPath('data/indicators/temp-annual-max-degree-over-20/historical/ukesm1-0-ll/ukesm1-0-ll_historical_temp-annual-max-degree-over-20_annual_1850_2014.nc'),
 PosixPath('data/indicators/temp-annual-max-degree-over-20/ssp126/gfdl-esm4/gfdl-esm4_ssp126_temp-annual-max-degree-over-20_annual_2015_2100.nc'),
 PosixPath('data/indicators/temp-annual-max-degree-over-20

## Option 3: building a local database file

To use an indicator whose files already exist locally (rather than downloading them from ISIMIP), we need to tell RIME-X where to find them. This is done via a **db file** -- a JSON file listing every simulation file, tagged with the specifiers (`climate_forcing`, `climate_scenario`, etc.) RIME-X uses to identify it.

Below we build one from the regridded heating-degree-days (HDD) files we created earlier. More examples of db file generation for different indicators can be found in `rimeX/examples`.

Once created, the path to this db file needs to be set in the `isimip_meta` option of the indicator's config definition (as `isimip_meta.db_file`), so RIME-X knows to load simulations from it instead of querying ISIMIP.

The db file has the following structure:

In [11]:
import glob, json, os, re

# Filenames encode model, scenario, and the covered time range, e.g.:
# ipsl-cm6a-lr_r1i1p1f1_w5e5_ssp370_hdd-18_global_annual_2015_2100.nc
pat = re.compile(r"(?P<model>[a-z0-9\-]+)_r1i1p1f(?P<fnum>\d+)_w5e5_(?P<scenario>[a-z0-9]+)_hdd-18_global_annual_(?P<start>\d{4})_(?P<end>\d{4})\.nc$")

# All regridded indicator files we want to register in the db.
# NB: use an absolute path here -- RIME-X joins this with `isimip.download_folder` to check
# whether the file already exists locally; a relative path here would fail that check and
# trigger an (unnecessary, and likely failing) download attempt from the real ISIMIP server.
files = glob.glob(os.path.abspath("data/raw_data/*.nc"))

# Build one db entry per file: "files" lists the path + time range,
# "specifiers" are the tags RIME-X uses to identify the simulation
db = [{"files": [{"time_slice": [int(m["start"]), int(m["end"])], "path": f}],
       "specifiers": {"climate_variable": "heating-degree-days", "climate_forcing": m["model"],
                       "climate_scenario": m["scenario"], "simulation_round": "isimip3b", "time_step": "annual"}}
      for f in files if (m := pat.search(os.path.basename(f)))]  # skip files that don't match the pattern

# Write the db file -- this path is what you set in isimip_meta.db_file
json.dump(db, open("data/heating-degree-days.json", "w"), indent=2)
print(f"{len(db)} entries written")

16 entries written


With the db file written, we can now define the indicator's config, pointing `isimip_meta.db_file` at it:

In [12]:
CONFIG['indicator.heating-degree-days'] = {
    'frequency': 'annual',
    'units': 'days*°C',
    'isimip_meta': {
        'db_file': 'data/heating-degree-days.json'
    }
}

Now this indicator can be "downloaded" too -- since the files already exist locally, this step doesn't fetch anything from the network, it just copies/links them into the format expected by the rest of the preprocessing pipeline.

In [13]:
heating_degree_days = Indicator.from_config("heating-degree-days")
heating_degree_days.run_download(dry_run=False)

[14:02:56 | rimeX | INFO] [1/16] heating-degree-days: downloading {'climate_scenario': 'historical', 'climate_forcing': 'gfdl-esm4'} ...
[14:02:56 | rimeX | INFO] [1/16] heating-degree-days: done -> /home/schwind/rimeX-with-socioeconomic-conditions/rimeX/tutorial/data/raw_data/gfdl-esm4_r1i1p1f1_w5e5_historical_hdd-18_global_annual_1850_2014.nc
[14:02:56 | rimeX | INFO] [2/16] heating-degree-days: downloading {'climate_scenario': 'historical', 'climate_forcing': 'mpi-esm1-2-hr'} ...
[14:02:56 | rimeX | INFO] [2/16] heating-degree-days: done -> /home/schwind/rimeX-with-socioeconomic-conditions/rimeX/tutorial/data/raw_data/mpi-esm1-2-hr_r1i1p1f1_w5e5_historical_hdd-18_global_annual_1850_2014.nc
[14:02:56 | rimeX | INFO] [3/16] heating-degree-days: downloading {'climate_scenario': 'historical', 'climate_forcing': 'mri-esm2-0'} ...
[14:02:56 | rimeX | INFO] [3/16] heating-degree-days: done -> /home/schwind/rimeX-with-socioeconomic-conditions/rimeX/tutorial/data/raw_data/mri-esm2-0_r1i1p1f1

[PosixPath('/home/schwind/rimeX-with-socioeconomic-conditions/rimeX/tutorial/data/raw_data/gfdl-esm4_r1i1p1f1_w5e5_historical_hdd-18_global_annual_1850_2014.nc'),
 PosixPath('/home/schwind/rimeX-with-socioeconomic-conditions/rimeX/tutorial/data/raw_data/mpi-esm1-2-hr_r1i1p1f1_w5e5_historical_hdd-18_global_annual_1850_2014.nc'),
 PosixPath('/home/schwind/rimeX-with-socioeconomic-conditions/rimeX/tutorial/data/raw_data/mri-esm2-0_r1i1p1f1_w5e5_historical_hdd-18_global_annual_1850_2014.nc'),
 PosixPath('/home/schwind/rimeX-with-socioeconomic-conditions/rimeX/tutorial/data/raw_data/ukesm1-0-ll_r1i1p1f2_w5e5_historical_hdd-18_global_annual_1850_2014.nc'),
 PosixPath('/home/schwind/rimeX-with-socioeconomic-conditions/rimeX/tutorial/data/raw_data/gfdl-esm4_r1i1p1f1_w5e5_ssp126_hdd-18_global_annual_2015_2100.nc'),
 PosixPath('/home/schwind/rimeX-with-socioeconomic-conditions/rimeX/tutorial/data/raw_data/mpi-esm1-2-hr_r1i1p1f1_w5e5_ssp126_hdd-18_global_annual_2015_2100.nc'),
 PosixPath('/home/s

## Next steps

At this point, all three indicators (`temp-annual-max`, `temp-annual-max-degree-over-20`, `heating-degree-days`) are downloaded/registered and ready for the next stage of preprocessing -- computing regional averages and then quantile maps. See `002_preprocessing_regional_aggregation.ipynb` and `003_preprocessing_quantilemaps.ipynb` in this tutorial series for those steps (only for the included `heating-degree-days` indicator, which is regridded to 5 degrees for storage considerations).

Once the quantile maps are obtained, you can use SCM (Simple Climate Model) data for your scenarios of interest to run emulations -- see `004_emulations.ipynb`. If you still need to obtain SCM data for your scenarios of interest, see `005_run_FaIR.ipynb` first.

## Appendix: additional indicator config options

The examples above only use a handful of config keys. `config.toml` supports quite a few more, useful for more complex indicators (see `[indicator.*]` entries in `config.toml` and the functions in `rimeX/indicators.py` for many worked examples). Here's a reference of what's available inside an `[indicator.<name>]` block.

**General**

| Key | Type | Description |
|---|---|---|
| `frequency` | str | Output time frequency, e.g. `"annual"`, `"monthly"` |
| `units` | str | Units of the indicator, e.g. `"%"`, `"°C"`, `"days*°C"` |
| `title` | str | Human-readable display title, e.g. `"Wet Bulb Temperature"` |
| `comment` | str | Free-text documentation, e.g. the formula/method used and a source reference |
| `folder` | str | Overrides `indicators.folder` just for this indicator |
| `year_min` | int | Overrides `isimip.historical_year_min` just for this indicator |
| `_copy` | str | Copy another indicator's config as a base, then override/add specific keys on top (see `wet_bulb_temperature_q95`, which copies `wet_bulb_temperature`) |

**Spatial & temporal aggregation**

| Key | Type | Description |
|---|---|---|
| `spatial_aggregation` | list[str] | Which regional weighting scheme(s) to use, e.g. `["latWeight"]` or `["latWeight", "pop2015"]`. Overrides the default list in `preprocessing.regional.weights` |
| `time_aggregation` | str | How to aggregate over a running window, e.g. `"max"` |
| `projection_baseline` | [int, int] | Overrides the global baseline period (`preprocessing.projection_baseline`) for this indicator, e.g. `[1995, 2014]` |
| `transform` | str | How the emulator expresses the indicator relative to its baseline: `"baseline_change"` (absolute difference) or `"baseline_change_percent"` (relative difference) |

**Deriving an indicator from other indicators**

| Key | Type | Description |
|---|---|---|
| `depends_on` | list[str] | Names of other (already-defined) indicators this one is computed from |
| `depends_on_climatology` | bool | Whether the computation additionally needs a historical climatology of the dependency/dependencies |
| `climatology_quantile` | float | If set, the climatology is a quantile (via `cdo timmin`/`timmax`/`timpctl`) rather than the default monthly mean, e.g. `0.999` for an extreme-value climatology |
| `climatology_manual` | bool | Use a raw concatenation of historical files as the "climatology" instead of any aggregation (you compute the statistic yourself in the custom function) |
| `custom` | str | `"module:function"` reference to a Python function that computes the indicator (see `rimeX/indicators.py`) |
| `expr` | str | A `cdo expr` formula evaluated directly on the (merged) dependency file(s), e.g. the wet-bulb-temperature approximation in `wet_bulb_temperature` |
| `shell` | str | An arbitrary shell/CDO command template, with placeholders `{input}`, `{inputs}`, `{output}`, `{previous_input}`, `{previous_output}`, `{name}` |

**Locating source data (`isimip_meta` sub-table)**

| Key | Type | Description |
|---|---|---|
| `variable` | str | The ISIMIP variable name to query, if different from the indicator name |
| `specifiers` | list[str] | Extra fields needed to uniquely identify the dataset on ISIMIP, e.g. `["variable", "crop"]` |
| `crop`, `soc_scenario`, `sens_scenario`, ... | str / list | Values for any extra specifiers declared above, e.g. `crop = "mai"` for maize |
| `ensemble_specifiers` | list[str] | Extra keys (beyond scenario/forcing) needed to identify an individual ensemble member, e.g. `["model"]` |
| `db_file` | str | Path to a local JSON db file (as built earlier in this notebook), used instead of querying ISIMIP directly |
| `year_min` | int | Same as the top-level `year_min`, but scoped inside `isimip_meta` |

**A note on custom functions**

A `custom` function (declared via `custom = "module:function"`) is called by RIME-X with a specific signature, depending on `depends_on_climatology`:

```python
# depends_on_climatology = false (or unset)
def my_indicator(input_files, output_file, previous_input_files=None, previous_output_file=None, dry_run=False, **kw):
    ...

# depends_on_climatology = true
def my_indicator(input_files, climatology_files, output_file, previous_input_files=None, previous_output_file=None, dry_run=False, **kw):
    ...
```

- `input_files` -- list of input files for the current time slice (one per entry in `depends_on`, or one raw ISIMIP file if `depends_on` is unset)
- `climatology_files` -- only passed if `depends_on_climatology = true`; the climatology file(s) computed from the historical period
- `output_file` -- where the function must write its result
- `previous_input_files` / `previous_output_file` -- the equivalent files from the previous time slice, useful for indicators needing continuity across slice boundaries (e.g. `rx5day`, which needs the last few days of the previous slice)

See `rimeX/indicators.py` for many working examples of each pattern (`cdo`-based, pure-`xarray`, with/without climatology, with/without `previous_*` continuity).